# 01 — Ingestion demo

Walks through the worker pipeline interactively. Run after `make seed` or after dropping an audio file into `data/audio/`.

## Set up paths and force the in-process settings

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
for sub in ('packages/core', 'packages/ml', 'services/worker'):
    p = os.path.join(ROOT, sub)
    if p not in sys.path: sys.path.insert(0, p)
os.environ.setdefault('DATABASE_URL', 'sqlite:///../echomind.db')
os.environ.setdefault('QDRANT_MODE', 'local')
os.environ.setdefault('QDRANT_PATH', '../qdrant_storage')
print('configured')

## Ingest a text-only book (no audio needed)

In [ ]:
from echomind_core.db import create_all
from echomind_worker.ingest import ingest_text
create_all()
result = ingest_text(
    slug='moby-dick-sample',
    title='Moby-Dick — sample',
    author='Herman Melville',
    genre='adventure',
    chapters_text=[
        'Call me Ishmael. Some years ago, never mind how long precisely.',
        'There she blows! There she blows! A hump like a snow-hill!',
    ],
)
result

## Confirm the chapters and embeddings landed

In [ ]:
from echomind_core.db import session_scope
from echomind_core.models import Book, Chapter
from sqlalchemy import select
with session_scope() as s:
    book = s.scalars(select(Book).where(Book.slug == 'moby-dick-sample')).first()
    chapters = s.scalars(select(Chapter).where(Chapter.book_id == book.id)).all()
    print('book:', book.title)
    for c in chapters:
        print(f'  chapter {c.idx}: {c.sentiment_label} ({c.sentiment_score:.2f})')

## Search for it

In [ ]:
from echomind_ml.embed_text import TextEmbedder
from echomind_core.vector import VectorStore, TEXT_COLLECTION
emb = TextEmbedder()
store = VectorStore()
hits = store.search(TEXT_COLLECTION, emb.embed_one('whale hunting'), k=3)
for h in hits:
    print(f'{h.score:.3f}  {h.payload.get("title")}  {h.payload.get("text")[:80]}')